In [2]:
import pandas as pd

# Load all three files
templates = pd.read_csv('preprocessed/HDFS.log_templates.csv')
labels    = pd.read_csv('preprocessed/anomaly_label.csv')
traces    = pd.read_csv('preprocessed/Event_traces.csv')

# Check shapes
print(templates.shape)  # (N_event_types, 3)
print(labels.shape)     # (575061, 2)
print(traces.shape)     # (575061, 2)

# Preview
print(traces.head())
print(labels.head())
print(templates.head())

(29, 2)
(575061, 2)
(575061, 6)
                    BlockId    Label  Type  \
0  blk_-1608999687919862906  Success   NaN   
1   blk_7503483334202473044  Success   NaN   
2  blk_-3544583377289625738     Fail  21.0   
3  blk_-9073992586687739851  Success   NaN   
4   blk_7854771516489510256  Success   NaN   

                                            Features  \
0  [E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26...   
1  [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   
2  [E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E3,E26,E26,...   
3  [E5,E22,E5,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   
4  [E5,E5,E22,E5,E11,E9,E11,E9,E11,E9,E26,E26,E26...   

                                        TimeInterval  Latency  
0  [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...     3802  
1  [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...     3802  
2  [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...     3797  
3  [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...    50448  
4  [0.0, 0.0, 1.0, 48.0, 0.0, 0.0,

In [4]:
import pandas as pd
import ast
from transformers import BertTokenizer

# ── 1. Load files ──────────────────────────────────────────
templates  = pd.read_csv('preprocessed/HDFS.log_templates.csv')
labels     = pd.read_csv('preprocessed/anomaly_label.csv')
traces     = pd.read_csv('preprocessed/Event_traces.csv')

# ── 2. Build EventId → Template text mapping ───────────────
event_map = dict(zip(templates['EventId'], templates['EventTemplate']))
# e.g. {'E1': 'Adding an already existing block', 'E5': 'Receiving block src dest', ...}

# ── 3. Parse Features string → actual list ─────────────────
traces['Features'] = traces['Features'].apply(lambda x: x.strip('[]').split(',') if isinstance(x, str) else x)

# ── 4. Merge traces with ground truth labels ───────────────
df = traces[['BlockId', 'Features']].merge(labels, on='BlockId')
# Now df has: BlockId | Features (list) | Label (Normal/Anomaly)

print(df.shape)
print(df['Label'].value_counts())
# Should show class distribution

# ── 5. Convert event IDs → template text sequences ─────────
def events_to_text(event_list):
    return " [SEP] ".join([event_map.get(e, "[UNK]") for e in event_list])

df['text'] = df['Features'].apply(events_to_text)

# Example output:
# "Receiving block src dest [SEP] Receiving block src dest [SEP] 
#  Got exception while serving [SEP] ..."

print(df['text'].iloc[2])  # check an anomaly

# ── 6. Binary label encoding ───────────────────────────────
df['label'] = (df['Label'] == 'Anomaly').astype(int)
# Normal → 0, Anomaly → 1

# ── 7. Sliding window ──────────────────────────────────────
def sliding_window(event_list, window=20, stride=10):
    windows = []
    for i in range(0, max(1, len(event_list) - window + 1), stride):
        windows.append(event_list[i:i + window])
    return windows

df['windows'] = df['Features'].apply(lambda x: sliding_window(x))

# Explode windows — each window becomes its own row
df_windows = df.explode('windows').reset_index(drop=True)
df_windows['window_text'] = df_windows['windows'].apply(
    lambda w: " [SEP] ".join([event_map.get(e, "[UNK]") for e in w])
)

print(f"Total windows: {df_windows.shape[0]}")
print(f"Anomaly windows: {df_windows['label'].sum()}")

# ── 8. Tokenize ────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(texts, max_length=512):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

# Test on first 5
sample = tokenize_batch(df_windows['window_text'].iloc[:5].tolist())
print(sample['input_ids'].shape)  # should be (5, 512)

(575061, 3)
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64
[*]Receiving block[*]src:[*]dest:[*] [SEP] [*]BLOCK* NameSystem[*]allocateBlock:[*] [SEP] [*]Receiving block[*]src:[*]dest:[*] [SEP] [*]Receiving block[*]src:[*]dest:[*] [SEP] [*]PacketResponder[*]for block[*]terminating[*] [SEP] [*]Received block[*]of size[*]from[*] [SEP] [*]PacketResponder[*]for block[*]terminating[*] [SEP] [*]Received block[*]of size[*]from[*] [SEP] [*]PacketResponder[*]for block[*]terminating[*] [SEP] [*]Received block[*]of size[*]from[*] [SEP] [*]Served block[*]to[*] [SEP] [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] [SEP] [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] [SEP] [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] [SEP] [*]Served block[*]to[*] [SEP] [*]Served block[*]to[*] [SEP] [*]Served block[*]to[*] [SEP] [*]Served block[*]to[*] [SEP] [*]Served block[*]to[*] [SEP] [*]Served

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\ark24\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ark24\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

torch.Size([5, 512])


In [6]:
import pandas as pd
import ast

# Load
templates = pd.read_csv('preprocessed/HDFS.log_templates.csv')
labels    = pd.read_csv('preprocessed/anomaly_label.csv')
traces    = pd.read_csv('preprocessed/Event_traces.csv')

# Parse Features string → actual list
traces['Features'] = traces['Features'].apply(lambda x: x.strip('[]').split(',') if isinstance(x, str) else x)

# Merge traces with ground truth labels
df = traces[['BlockId', 'Features']].merge(labels, on='BlockId')

# Check
print("Dataset shape:", df.shape)
print("\nLabel distribution:")
print(df['Label'].value_counts())

# Build event map
event_map = dict(zip(templates['EventId'], templates['EventTemplate']))

# Convert event IDs → text
def events_to_text(event_list):
    return " [SEP] ".join([event_map.get(e, "[UNK]") for e in event_list])

df['text'] = df['Features'].apply(events_to_text)

# Binary label
df['label'] = (df['Label'] == 'Anomaly').astype(int)

# Sliding window
def sliding_window(event_list, window=20, stride=10):
    windows = []
    for i in range(0, max(1, len(event_list) - window + 1), stride):
        windows.append(event_list[i:i + window])
    return windows

df['windows'] = df['Features'].apply(sliding_window)

# Explode
df_windows = df.explode('windows').reset_index(drop=True)
df_windows['window_text'] = df_windows['windows'].apply(
    lambda w: " [SEP] ".join([event_map.get(e, "[UNK]") for e in w])
)

print("\nTotal windows:", df_windows.shape[0])
print("Anomaly windows:", df_windows['label'].sum())
print("\nSample anomaly text:")
print(df_windows[df_windows['label']==1]['window_text'].iloc[0])

Dataset shape: (575061, 3)

Label distribution:
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64

Total windows: 598766
Anomaly windows: 19596

Sample anomaly text:
[*]Receiving block[*]src:[*]dest:[*] [SEP] [*]BLOCK* NameSystem[*]allocateBlock:[*] [SEP] [*]Receiving block[*]src:[*]dest:[*] [SEP] [*]Receiving block[*]src:[*]dest:[*] [SEP] [*]PacketResponder[*]for block[*]terminating[*] [SEP] [*]Received block[*]of size[*]from[*] [SEP] [*]PacketResponder[*]for block[*]terminating[*] [SEP] [*]Received block[*]of size[*]from[*] [SEP] [*]PacketResponder[*]for block[*]terminating[*] [SEP] [*]Received block[*]of size[*]from[*] [SEP] [*]Served block[*]to[*] [SEP] [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] [SEP] [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] [SEP] [*]BLOCK* NameSystem[*]addStoredBlock: blockMap updated:[*]is added to[*]size[*] [SEP] [*]Served block[*]to[*] [SEP] [*]Served block[*]to[*] [

In [ ]:
import json

def convert_to_training_jsonl(input_path, output_path):
    """
    Converts hdfs_explanations.json to .jsonl fine-tuning format
    for Qwen2.5-7B (chat template compatible with most frameworks).
    """

    SYSTEM_PROMPT = (
        "You are an HDFS log analysis expert specializing in Apache Hadoop "
        "distributed storage systems. Given a block's event sequence, anomaly "
        "type, and latency, explain the anomaly in plain English. Your "
        "explanation must cover: (1) what events signal the anomaly and why "
        "their ordering is anomalous, (2) the likely root cause, and "
        "(3) the cluster-level impact. Write 3-5 sentences of natural "
        "technical prose. Do not use event ID codes like E5 or E22 in your "
        "explanation — translate them to plain English descriptions."
    )

    with open(input_path, "r") as f:
        data = json.load(f)

    with open(output_path, "w") as out:
        for entry in data:
            sequence_str = ", ".join(entry["sequence"])

            user_content = (
                f"Block: {entry['block_id']}\n"
                f"Anomaly Type: {entry['anomaly_type']}\n"
                f"Sequence: [{sequence_str}]\n"
                f"Sequence Length: {len(entry['sequence'])}"
            )

            # Add latency if available in your data
            # If you extend hdfs_explanations.json to include latency:
            if "latency" in entry:
                user_content += f"\nLatency: {entry['latency']}s"

            messages = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": entry["explanation"]}
                ]
            }

            out.write(json.dumps(messages, ensure_ascii=False) + "\n")

    print(f"Written {len(data)} training examples to {output_path}")


if __name__ == "__main__":
    convert_to_training_jsonl(r"D:\hdfs dataset\preprocessed\hdfs_explanations.json", "train.jsonl")

In [1]:
import json

def convert_to_training_jsonl(input_path, output_path):
    """
    Converts hdfs_explanations.json to .jsonl fine-tuning format
    for Qwen2.5-7B (chat template compatible with most frameworks).
    """

    SYSTEM_PROMPT = (
        "You are an HDFS log analysis expert specializing in Apache Hadoop "
        "distributed storage systems. Given a block's event sequence, anomaly "
        "type, and latency, explain the anomaly in plain English. Your "
        "explanation must cover: (1) what events signal the anomaly and why "
        "their ordering is anomalous, (2) the likely root cause, and "
        "(3) the cluster-level impact. Write 3-5 sentences of natural "
        "technical prose. Do not use event ID codes like E5 or E22 in your "
        "explanation — translate them to plain English descriptions."
    )

    with open(input_path, "r") as f:
        data = json.load(f)

    with open(output_path, "w") as out:
        for entry in data:
            sequence_str = ", ".join(entry["sequence"])

            user_content = (
                f"Block: {entry['block_id']}\n"
                f"Anomaly Type: {entry['anomaly_type']}\n"
                f"Sequence: [{sequence_str}]\n"
                f"Sequence Length: {len(entry['sequence'])}"
            )

            # Add latency if available in your data
            # If you extend hdfs_explanations.json to include latency:
            if "latency" in entry:
                user_content += f"\nLatency: {entry['latency']}s"

            messages = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": entry["explanation"]}
                ]
            }

            out.write(json.dumps(messages, ensure_ascii=False) + "\n")

    print(f"Written {len(data)} training examples to {output_path}")


if __name__ == "__main__":
    convert_to_training_jsonl(r"D:\hdfs dataset\preprocessed\hdfs_explanations.json", "train.jsonl")

Written 16600 training examples to train.jsonl


In [1]:
import pandas as pd

traces = pd.read_csv("D:/hdfs_dataset/preprocessed/Event_traces.csv")
templates = pd.read_csv("D:/hdfs_dataset/preprocessed/HDFS.log_templates.csv")

print("Event_traces columns:", traces.columns.tolist())
print("First row:", traces.iloc[0].to_dict())
print()
print("Templates columns:", templates.columns.tolist())
print("First row:", templates.iloc[0].to_dict())
print()
print("Unique labels:", traces[traces.columns[-1]].unique()[:10])

Event_traces columns: ['BlockId', 'Label', 'Type', 'Features', 'TimeInterval', 'Latency']
First row: {'BlockId': 'blk_-1608999687919862906', 'Label': 'Success', 'Type': nan, 'Features': '[E5,E22,E5,E5,E11,E11,E9,E9,E11,E9,E26,E26,E26,E6,E5,E16,E6,E5,E18,E25,E26,E26,E3,E25,E6,E6,E5,E5,E16,E18,E26,E26,E5,E6,E5,E16,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E18,E25,E6,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E26,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E25,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E18,E6,E5,E3,E3,E3,E3,E3,E16,E3,E3,E3,E3,E26,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E3,E23,E23,E23,E23,E23,E23,E23,E23,E23,E23

In [2]:
"""
LogSense — Combine Datasets
----------------------------
Combines logsense_clean_train.jsonl + logsense_generated.jsonl
Deduplicates on output text
Saves to logsense_final_train.jsonl + logsense_final_val.jsonl
"""

import json
import random
from collections import Counter

FILE_1    = "D:/hdfs_dataset/logsense_clean_train.jsonl"   # 382 clean records
FILE_2    = "D:/hdfs_dataset/logsense_generated.jsonl"     # 203 generated records
OUT_TRAIN = "logsense_final_train.jsonl"
OUT_VAL   = "logsense_final_val.jsonl"
VAL_SPLIT = 0.1
SEED      = 42


def load(path: str) -> list[dict]:
    records = []
    try:
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        print(f"  Loaded {len(records):>4} records from {path}")
    except FileNotFoundError:
        print(f"  MISSING: {path} — skipping")
    return records


def get_output(rec: dict) -> str:
    return rec["messages"][2]["content"]


def get_anomaly_type(rec: dict) -> str:
    user = rec["messages"][1]["content"]
    for line in user.split("\n"):
        if line.startswith("Anomaly Type:"):
            return line.split(":", 1)[1].strip()
    return "unknown"


def main():
    print("Loading files...")
    records_1 = load(FILE_1)
    records_2 = load(FILE_2)

    all_records = records_1 + records_2
    print(f"  Total before dedup: {len(all_records)}")

    # ── Deduplicate on exact output text ──────────────────────────────────────
    seen = set()
    unique = []
    for rec in all_records:
        out = get_output(rec)
        if out not in seen:
            seen.add(out)
            unique.append(rec)

    print(f"  Total after dedup : {len(unique)}")
    print(f"  Duplicates removed: {len(all_records) - len(unique)}")

    # ── Type distribution ─────────────────────────────────────────────────────
    type_dist = Counter(get_anomaly_type(r) for r in unique)
    print("\n  Anomaly type distribution:")
    for t, c in sorted(type_dist.items(), key=lambda x: -x[1]):
        print(f"    Type {t:<6} {c:>4} samples")

    # ── Train / val split ─────────────────────────────────────────────────────
    random.seed(SEED)
    random.shuffle(unique)

    split     = int(len(unique) * (1 - VAL_SPLIT))
    train     = unique[:split]
    val       = unique[split:]

    # ── Save ──────────────────────────────────────────────────────────────────
    with open(OUT_TRAIN, "w") as f:
        for rec in train:
            f.write(json.dumps(rec) + "\n")

    with open(OUT_VAL, "w") as f:
        for rec in val:
            f.write(json.dumps(rec) + "\n")

    print(f"\n{'═'*45}")
    print(f"  Train : {len(train)} records → {OUT_TRAIN}")
    print(f"  Val   : {len(val)} records → {OUT_VAL}")
    print(f"{'═'*45}")
    print("\n  Ready for fine-tuning.")


if __name__ == "__main__":
    main()

Loading files...
  Loaded  425 records from D:/hdfs_dataset/logsense_clean_train.jsonl
  Loaded  203 records from D:/hdfs_dataset/logsense_generated.jsonl
  Total before dedup: 628
  Total after dedup : 628
  Duplicates removed: 0

  Anomaly type distribution:
    Type 0       136 samples
    Type 5       135 samples
    Type 7       105 samples
    Type 1       103 samples
    Type 31       62 samples
    Type 21       29 samples
    Type 4        24 samples
    Type 16       20 samples
    Type 9         7 samples
    Type 12        3 samples
    Type 30        1 samples
    Type 27        1 samples
    Type 8         1 samples
    Type 22        1 samples

═════════════════════════════════════════════
  Train : 565 records → logsense_final_train.jsonl
  Val   : 63 records → logsense_final_val.jsonl
═════════════════════════════════════════════

  Ready for fine-tuning.
